# Execution modes

### Development workflow

The term **Qiskit pattern** describes the development workflow for breaking down domain-specific problems and contextualizing required capabilities in stages. This allows for the seamless composability of new capabilities developed by IBM Quantum® researchers (and others) and enables a future in which quantum computing tasks are performed by powerful heterogenous (CPU/GPU/QPU) computing infrastructure. Blocks or groups of blocks perform the steps of a pattern, with the Qiskit SDK providing an important foundational layer, supported by other tools or services developed by IBM Quantum or the quantum open-source community. Qiskit patterns allow domain experts to specify a problem and compose the tooling (blocks) that achieves a Qiskit pattern. That pattern can then be executed locally, through cloud services, or deployed with Qiskit Serverless.

The four steps of a Qiskit pattern are as follows:

- ##### **Map problem to quantum circuits and operators**

        This step describes how a user starts with a classical problem and figures out how to map it to a quantum computer. For example, in applications such as chemistry and quantum simulation, this step generally involves constructing a quantum circuit representing the Hamiltonian you are attempting to solve. The output of this step is normally a collection of circuits or quantum operators that can be optimized for hardware in the next step.
- ##### **Optimize for target hardware**

        In this step you take the abstract circuits (or operators) produced from the map step and perform a series of optimizations on them. This can include mapping the route and layout of the circuit to physical qubit hardware, converting to basis gates of the hardware, and reducing the number of operations, all designed to optimize the likelihood of success in the later execute step. During this step, abstract circuits must be transpiled to **Instruction Set Architecture (ISA)** circuits. An ISA circuit is one that only consists of gates understood by the target hardware (basis gates), and any multi-qubit gates needed to obey any connectivity constraints (coupling map). Only ISA circuits can be run on IBM hardware using IBM Qiskit Runtime.



- ##### **Execute on target hardware**

        This step involves running your circuits on hardware and produces the outputs of the quantum computation. The ISA circuits produced in the previous step can be executed using either a Sampler or Estimator primitive from Qiskit Runtime, initialized locally on your computer or from a cluster or other heterogeneous compute environment. These can be executed in a Batch, which allows parallel transpilation for classical computational efficiency - or a Session, which allows iterative tasks to be implemented efficiently without queuing delays. During this step, there is also the option to configure certain error suppression and mitigation techniques provided by Qiskit Runtime. Depending on whether you are using the Sampler or Estimator primitive, the outcome of this step will be different. If using the Sampler, the output will be per-shot measurements in the form of bitstrings. If using the Estimator, the output will be expectation values of observables corresponding to physical quantities or cost functions.

- ##### **Post-process result**

        This final step involves stitching the outputs from the prior step back together to obtain the desired result. This can involve a range of classical data-processing steps such as visualizing results, readout error mitigation techniques, marginalizing quasi-probability distributions to ascertain results on smaller sets of qubits, or post-selection on inherent properties of the problem, such as total spin, parity, or particle conservation by removing unphysical observable



### Execution mode: Types

There are three execution modes: **job, session, and batch.**

### Job mode

A single primitive request of the estimator or the sampler made without a context manager. Circuits and inputs are packaged as primitive unified blocs (PUBs) and submitted as an execution task on the quantum computer. To run in job mode, specify `mode=backend` when instantiating a primitive.


#### Single circuit in job mode (`Estimator` primitive)

In [ ]:
import numpy as np
from qiskit.circuit.library import iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp, random_hermitian
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as Estimator

n_qubits = 50

# Running using Esimator primitive in Job mode

service = QiskitRuntimeService(name= "default-ibm-quantum-platform") # use your IBM Quantum instance name here
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=n_qubits
)

mat = np.real(random_hermitian(n_qubits, seed=1234)) # generate a random Hermitian matrix as the input to the iqp circuit
circuit = iqp(mat) 
observable = SparsePauliOp("Z" * 50) # expectation value of ZZZ...Z (on all 50 qubits)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(circuit)
isa_observable = observable.apply_layout(isa_circuit.layout) # apply the layout of the circuit to the observable so that they are compatible for execution on the backend

estimator = Estimator(mode=backend) 
job = estimator.run([(isa_circuit, isa_observable)]) # job mode
result = job.result()

print(f" > Expectation value: {result[0].data.evs}")
print(f" > Metadata: {result[0].metadata}")

 > Expectation value: 0.06779661016949153
 > Metadata: {'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32}


#### Multiple circuits in job mode (`Estimator` primitive)

In [10]:
rng = np.random.default_rng()
mats = [np.real(random_hermitian(n_qubits, seed=rng)) for _ in range(3)]

pubs = []
circuits = [iqp(mat) for mat in mats]
observables = [
    SparsePauliOp("X" * 50),
    SparsePauliOp("Y" * 50),
    SparsePauliOp("Z" * 50),
]

# Get ISA circuits
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)

for qc, obs in zip(circuits, observables):
    isa_circuit = pm.run(qc)
    isa_obs = obs.apply_layout(isa_circuit.layout)
    pubs.append((isa_circuit, isa_obs))

estimator = Estimator(backend)
job = estimator.run(pubs) # job mode with multiple PUBs
job_result = job.result()

for idx in range(len(pubs)):
    pub_result = job_result[idx]
    print(f">>> Expectation values for PUB {idx}: {pub_result.data.evs}")
    print(f">>> Standard errors for PUB {idx}: {pub_result.data.stds}")

>>> Expectation values for PUB 0: -0.050156739811912224
>>> Standard errors for PUB 0: 0.22666245619005568
>>> Expectation values for PUB 1: 0.27817745803357313
>>> Standard errors for PUB 1: 0.3190941854914009
>>> Expectation values for PUB 2: -1.1216931216931216
>>> Standard errors for PUB 2: 0.6659716020234819


#### Single circuit in job mode (`Sampler` primitive)

In [ ]:
from qiskit_ibm_runtime import SamplerV2 as Sampler

n_qubits = 127
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=n_qubits
)

mat = np.real(random_hermitian(n_qubits, seed=1234))
circuit = iqp(mat)
circuit.measure_all()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(circuit)

sampler = Sampler(backend)
job = sampler.run([isa_circuit])
result = job.result()

# Get results for the first (and only) PUB
pub_result = result[0]

print(f" > First ten results: {pub_result.data.meas.get_bitstrings()[:10]}") # meas is valid if measure_all used, else name of the classical register containing measurement results needs to be specified

 > First ten results: ['1000010001001001010001001111000110101001010111001001111010001110011110001000010010011100011001010100100011000001101000000000101', '1001110111110111000000000001000101000110010101000100110000000100000000011000010001000011010101101110111000100001011001000000000', '1101110001011110010110110001101100001001010000110001010000101111001101011000100000010101110111000001101110001010010000001000101', '0111111001011111110110100010000101001001010101101000000001100110001100010001101000101001100101000010000000000011100011000100000', '1111110001110110011111100111100101110001010001000010000010100000000100011001000100010001011111111001100000100000010010010010000', '1100101101011000000011011010110011100010110100000000001111001111001011010001000010110100100011010001000000100000000000011100100', '1101110000100001111010010110001000010101000110011000101000101001000011010100001011001010001001001010000100000000010000010011010', '00111010101110010001010100110011001001001101001000010011000

#### Multiple circuits in job mode (`Sampler` primitive)

In [12]:
rng = np.random.default_rng()
mats = [np.real(random_hermitian(n_qubits, seed=rng)) for _ in range(3)]
circuits = [iqp(mat) for mat in mats]
for circuit in circuits:
    circuit.measure_all()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuits = pm.run(circuits)

sampler = Sampler(mode=backend)
job = sampler.run(isa_circuits)
result = job.result()

for idx, pub_result in enumerate(result):
    print(
        f" > First ten results for pub {idx}: {pub_result.data.meas.get_bitstrings()[:10]}"
    )

 > First ten results for pub 0: ['1011101000011001001100010001111000110011001101010011110101100010010010001010000010000111110101000000000010001000011010101111101', '0000101000011010101001011110010010100010110110001110010100111010100110000110110010101110101101100001000000001011101001110001000', '0101000001111111110101001000001011110111000001000001100000101000000111000100000010001000101111100000011000101100101010110001001', '1100110100001110000010001100001100101001000001111001011010010000100000001100011001111111010010111101110100100111111001100011001', '0100000110000101100100101111110101000111000010011110000000001111011110000010100010000001001000010001001110101000100000101111100', '1100101100010001100100110101010111010111000111000101001110111010001100000000000000101100110001010100011001000000111011110101011', '1111110000101000101010000010010110000100101100000111111001101100000001100111001001101001010001101101000000100010001001001000100', '1110101010010000001000100011000010101111111110001

### Batch mode

A multi-job manager for efficiently running experiments comprising multi-job workloads. These workloads are made up of independently executable jobs that have no conditional relationship with each other. With batch mode, users submit their jobs all at once.

The system parallelizes or threads the pre-processing (classical computing) step of each primitive job to more tightly package quantum execution across jobs, and then runs the quantum execution of each job in quick succession to deliver the most efficient results. 

#### Note
- When batching, jobs are not guaranteed to run in the order they are submitted. Also, while your batch jobs will run as closely together as possible, they don't get exclusive access to the backend. Therefore, your batch jobs might run in parallel with other users' jobs if there is enough processing capacity on the QPU. Additionally, QPU calibration jobs could run between the batched jobs.
- The queuing time does not decrease for the first job submitted within a batch. Therefore, batches do not provide any benefits when running a single job.

To run in batch mode, specify `mode=batch` when instantiating a primitive or run the job in a batch context manager.

<div style="display: flex; justify-content: center;">
  <img src="Batch.png" style="width: 50%; height: auto;">
</div>

____



In [ ]:
from qiskit_ibm_runtime import Batch

backend = service.least_busy(operational=True, simulator=False)
batch = Batch(backend=backend)
estimator = Estimator(mode=batch)
sampler = Sampler(mode=batch)

job = sampler.run(isa_circuits) # same sampler job but in batch mode
result = job.result()

for idx, pub_result in enumerate(result):
    print(
        f" > First ten results for pub {idx}: {pub_result.data.meas.get_bitstrings()[:10]}"
    )

batch.close()

 > First ten results for pub 0: ['0101101100101100000111100101001011111110001110011100100000011101111100100001101100110110000000011000010000000100101110110101000', '1111100111101001001100000001100001010101001001011001001110100101000001110000000100010010000110110110010111000101010000000001100', '1100011010000000101100001111100110011100011111010111010000010110010011000101011010001100100100001101100011000101100000000001000', '0111011001011110001010011111000111101101011100100000001100010010011010001000101100101100001110011000000001001111110010010010001', '0100010111111100110000111000011110100100011110000011010010101000000100000100011010010010000001101111001001011001101010000000000', '0111010010001011100001000011100000100010101010110010010101000000011100100000001101100001101010100000110001000001101011110010110', '1100010000001000110101001000110000000001101010000011010110100000011001101001110000010010110010000000000111000010001001001100000', '1111000100111110110100000101100000011101000101001

- **Context manager**

The context manager automatically opens and closes the batch.

A batch automatically closes when it exits the context manager. When the batch context manager is exited, the batch is put into "In progress, not accepting new jobs" status. This means that the batch finishes processing all running or queued jobs until the maximum TTL value is reached. After all jobs are completed, the batch is immediately closed. You cannot submit jobs to a closed batch.

- **Batch Length**

You can define the batch's maximum time to live (TTL) with the max_time parameter. This should exceed the longest job's execution time. This timer starts when the batch starts. When the value is reached, the batch is closed. Any jobs that are running will finish, but jobs still queued are failed.



In [ ]:

with Batch(backend=backend): # this is the context manager way of running in batch mode, which will automatically close the batch at the end of the block
    estimator = Estimator() # note no mode is passed here
    sampler = Sampler()
    # Run estimator and sampler jobs in batch mode as needed, e.g. estimator.run(pubs) or sampler.run(isa_circuits)

with Batch(backend=backend, max_time="25m"): # Max TTL specified for the batch
    estimator = Estimator()
    sampler = Sampler()
    # Run estimator and sampler jobs in batch mode as needed, e.g. estimator.run(pubs) or sampler.run(isa_circuits)

In [14]:
# printing batch details 

backend = service.least_busy(operational=True, simulator=False)

with Batch(backend=backend) as batch:
    print(batch.details())

{'id': 'bb605488-dd33-409b-841b-3275bb0d6b63', 'backend_name': 'ibm_fez', 'interactive_timeout': 1, 'max_time': 600, 'active_timeout': 600, 'state': 'open', 'accepting_jobs': True, 'last_job_started': None, 'last_job_completed': None, 'started_at': None, 'closed_at': None, 'activated_at': None, 'mode': 'batch', 'usage_time': None}


There are multiple ways you can reconfigure your jobs to take advantage of the parallel processing provided by batching. The following example shows how you can partition a long list of circuits into multiple jobs and run them as a batch to take advantage of the parallel processing.

In [ ]:
from qiskit.circuit.random import random_circuit

max_circuits = 100
circuits = [pm.run(random_circuit(5, 5)) for _ in range(5 * max_circuits)] # 500 circuits
for circuit in circuits:
    circuit.measure_active()
all_partitioned_circuits = [] 
for i in range(0, len(circuits), max_circuits):
    all_partitioned_circuits.append(circuits[i : i + max_circuits]) # 5 groups of 100 circuits each
jobs = []
start_idx = 0

with Batch(backend=backend): # a batch is opened here, and all jobs submitted within this block will be part of the same batch
    sampler = Sampler()
    for partitioned_circuits in all_partitioned_circuits:
        job = sampler.run(partitioned_circuits) # 100 circuits submitted as a single job
        jobs.append(job) # appending the job to a list so that we can retrieve results later

Note: In the code above:

**Without Batch:**
Each job goes through:
- queue
- scheduling
- execution
Backend sees independent requests

**With Batch:**

Backend sees:
"Here’s a bundle of jobs — optimize execution globally"

So it can:
- Run jobs on different QPUs / time slices
- Interleave circuits across jobs
- Reduce queue overhead
- Keep hardware busy continuously

<div style="display: flex; justify-content: auto;">
  <img src="Batch_faq_1.png" style="height: 220px; width: auto;">
</div>

_____


<div style="display: flex; justify-content: auto;">
  <img src="Batch_faq_2.png" style="width: 60%; height: auto;">
</div>

### Session mode

A dedicated window for running a multi-job workload. During this window, the user has exclusive access of the system and no other jobs can run - including calibration jobs. This allows users to experiment with variational algorithms in a more predictable way and even run multiple experiments simultaneously, taking advantage of parallelism in the stack. Using sessions helps avoid delays caused by queuing each job separately, which can be particularly useful for iterative tasks that require frequent communication between classical and quantum resources.

To run in session mode, specify `mode=session` when instantiating a primitive, or run the job in a session context manager.

The queuing time does not decrease for the first job submitted within a session. Therefore, sessions do not provide any benefits when running a single job.

<div style="display: flex; justify-content: center;">
  <img src="Session.png" style="width: 50%; height: auto;">
</div>

____




#### The basic workflow for batches and sessions is similar:

- The first job in a batch or session enters the normal queue. For batches, the entire batch of jobs is scheduled together.

- When the first job starts running, the maximum time to live (TTL) timer starts, and does not stop or pause until the end is reached.
The interactive TTL timer starts after each job is completed. If there are no workload jobs ready within the interactive TTL window, the workload is temporarily deactivated and normal job selection resumes. A job can reactivate the deactivated workload if the batch or session has not reached its maximum TTL value.
(The job must go through the normal queue to reactivate the workload.)
- If the maximum TTL value is reached, the workload ends and any remaining queued jobs fail. Any job currently running won't run to completion if doing so would exceed the instance's cost limit.

### Chosing the right execution mode

The benefits of each are summarized below:

#### **Batch**

- The entire batch of jobs is scheduled together and there is no additional queuing time for each.
- The jobs' classical computation, such as compilation, is run in parallel. Thus, running multiple jobs in a batch is significantly faster than running them serially.
- There is usually minimal delay between jobs, which can help avoid drift.
- If you partition your workload into multiple jobs and run them in batch mode, you can get results from individual jobs, which makes them more flexible to work with. For example, if a job's results don't meet your expectations, you can cancel the remaining jobs. Also, if one job fails, you can re-submit it instead of re-running the entire workload.
- Is generally less expensive than sessions.

#### **Session**

- All the functionality from batch mode (but requiring increased usage)
- Dedicated and exclusive access to the QPU during the session active window.
- Useful for workloads that don’t have all inputs ready at the outset, for iterative workloads that require classical post-processing before the next one can run, and for experiments that need to run as tightly together as possible.

#### **Job**
- Easiest to use when running a small experiment.
- Might run sooner than batch mode.


___
___